In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, roc_auc_score)

RANDOM_STATE = 42
TEST_SIZE = 0.20
sns.set_theme(style="whitegrid")
candidate_paths = [Path("raw_intents_data.csv"), Path("/mnt/data/raw_intents_data.csv")]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Upload raw_intents_data.csv next to this notebook or update candidate_paths.")
df = pd.read_csv(data_path)
print("Loaded:", data_path.resolve(), "| shape:", df.shape)
display(df.head())
df.info()

In [ ]:
required = {"text", "intent"}
if not required.issubset(df.columns):
    raise ValueError(f"Expected {required}; actual columns are {df.columns.tolist()}")
print("Missing values (%):\n", (df.isna().mean()*100).round(2))
print("Exact duplicates before cleaning:", int(df.duplicated().sum()))
df = df[["text", "intent"]].copy()
df["text"] = df["text"].astype("string").str.strip()
df["intent"] = df["intent"].astype("string").str.strip()
df = df.dropna(subset=["text", "intent"])
df = df[(df.text != "") & (df.intent != "")]
df = df.drop_duplicates(subset=["text", "intent"]).reset_index(drop=True)
conflicts = df.groupby("text").intent.nunique()
if (conflicts > 1).any():
    raise ValueError("Identical texts map to multiple labels; inspect before modelling.")
print("Clean shape:", df.shape)
display(df.intent.value_counts().rename_axis("intent").to_frame("count"))

X, y = df["text"], df["intent"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train/test:", len(X_train), len(X_test))

In [ ]:
def get_metrics(pipe, X_eval, y_eval, name, train_seconds, predict_seconds):
    pred = pipe.predict(X_eval)
    row = {
        "model": name,
        "accuracy": accuracy_score(y_eval, pred),
        "precision_macro": precision_score(y_eval, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_eval, pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_eval, pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_eval, pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_eval, pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_eval, pred, average="weighted", zero_division=0),
        "train_seconds": train_seconds,
        "predict_seconds": predict_seconds,
    }
    scores = pipe.predict_proba(X_eval) if hasattr(pipe, "predict_proba") else (pipe.decision_function(X_eval) if hasattr(pipe, "decision_function") else None)
    try:
        row["roc_auc_macro_ovr"] = roc_auc_score(y_eval, scores, labels=pipe.classes_, multi_class="ovr", average="macro") if scores is not None else np.nan
    except (ValueError, TypeError):
        row["roc_auc_macro_ovr"] = np.nan
    return row, pred

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
common_tfidf = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), sublinear_tf=True, max_features=5000)

In [ ]:
model_specs = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, solver="saga", random_state=RANDOM_STATE),
        {"classifier__C": [0.1, 1.0, 10.0], "classifier__l1_ratio": [0.0]},
        "logistic_regression_pipeline.joblib"
    ),
    "SVC": (
        SVC(kernel="linear", C=1.0, probability=True, random_state=RANDOM_STATE),
        {"classifier__kernel": ["linear", "rbf"], "classifier__C": [0.5, 1.0, 5.0], "classifier__gamma": ["scale"]},
        "svm_svc_pipeline.joblib"
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=1),
        {"classifier__n_estimators": [80, 120], "classifier__max_depth": [None, 30], "classifier__min_samples_leaf": [1], "classifier__max_features": ["sqrt"]},
        "random_forest_pipeline.joblib"
    ),
    "Multinomial Naive Bayes": (
        MultinomialNB(alpha=1.0),
        {"classifier__alpha": [0.1, 0.5, 1.0, 2.0]},
        "naive_bayes_pipeline.joblib"
    ),
}

results = []
trained_models = {}
predictions_by_model = {}
for name, (estimator, grid, filename) in model_specs.items():
    print(f"\n{'='*20} {name} {'='*20}")
    pipe = Pipeline([("tfidf", common_tfidf), ("classifier", estimator)])
    search = GridSearchCV(pipe, grid, scoring="f1_macro", cv=cv, n_jobs=1, refit=True, verbose=0)
    t0 = time.perf_counter()
    search.fit(X_train, y_train)
    train_seconds = time.perf_counter() - t0
    best_pipe = search.best_estimator_
    t0 = time.perf_counter()
    pred = best_pipe.predict(X_test)
    predict_seconds = time.perf_counter() - t0
    row, pred = get_metrics(best_pipe, X_test, y_test, name, train_seconds, predict_seconds)
    row["best_cv_f1_macro"] = search.best_score_
    row["best_params"] = str(search.best_params_)
    results.append(row)
    trained_models[name] = best_pipe
    predictions_by_model[name] = pred
    joblib.dump(best_pipe, filename)
    print("Best CV macro F1:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)
    print("Saved:", filename)
    print(classification_report(y_test, pred, zero_division=0))

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
display(results_df)
results_df.to_csv("model_comparison_results.csv", index=False)
print("Saved results to model_comparison_results.csv")

In [ ]:
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_macro_ovr"]
display(results_df.set_index("model")[metric_cols])

ax = results_df.set_index("model")[["accuracy", "precision_macro", "recall_macro", "f1_macro"]].plot(kind="bar", figsize=(11, 5))
ax.set_title("Held-out test performance comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
results_df.set_index("model")[["train_seconds", "predict_seconds"]].plot(kind="bar", ax=ax)
ax.set_title("Runtime comparison on this environment")
ax.set_ylabel("Seconds")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
error_rows = []
for name, pred in predictions_by_model.items():
    errors = pd.DataFrame({"actual": y_test.to_numpy(), "predicted": np.asarray(pred)})
    errors = errors[errors.actual != errors.predicted]
    if len(errors):
        counts = errors.groupby(["actual", "predicted"]).size().sort_values(ascending=False).head(10)
        for (actual, predicted), count in counts.items():
            error_rows.append({"model": name, "actual_intent": actual, "predicted_intent": predicted, "count": int(count)})
error_table = pd.DataFrame(error_rows)
if len(error_table):
    display(error_table.sort_values(["model", "count"], ascending=[True, False]))
else:
    print("No misclassifications on the held-out test set for any model.")

In [ ]:
best_name = results_df.iloc[0]["model"]
best_pipe = trained_models[best_name]
best_pred = predictions_by_model[best_name]
labels = best_pipe.classes_
cm = confusion_matrix(y_test, best_pred, labels=labels)
plt.figure(figsize=(11, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title(f"Confusion matrix — {best_name}")
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
print("Top model by held-out macro F1:", best_name)
print("This is a result for this split and dataset, not a universal ranking.")